In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

In [9]:
load_dotenv()

llm = ChatOpenAI()

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [10]:
def chat_node(state: ChatState):
    messages = state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}

In [11]:
checkpointer = InMemorySaver()

graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
for message_chunk, metadata in chatbot.stream(
    {"messages": [HumanMessage(content="What is the recipe to make pasta")]},
    config={"configurable": {"thread_id": "thread-1"}},
    stream_mode="messages"
):
    if message_chunk.content:
       print(message_chunk.content, end=" ", flush=True)

Ingredients :

 -   8  oz .  of  pasta  ( any  variety )
 -   4  cups  of  water 
 -   1  tsp  of  salt 
 -   1  tbsp  of  olive  oil 

 Instructions :

 1 .  In  a  large  pot ,  bring  the  water  to  a  boil .  Add  the  salt  and  olive  oil .
    
 2 .  Add  the  pasta  to  the  boiling  water  and  stir  occasionally  to  prevent  sticking .
    
 3 .  Cook  the  pasta  according  to  package  instructions ,  usually  around   8 - 10  minutes  for  al  d ente .
    
 4 .  Once  the  pasta  is  cooked ,  drain  it  in  a  col ander  and  rinse  with  cold  water  to  stop  the  cooking  process .
    
 5 .  Serve  the  pasta  with  your  favorite  sauce  or  toppings .  Enjoy ! 

: 